# 06 — Inference on a Spatial Test Block

Feeds a **single held-out test block** from the materialized spatial split
(`/Volumes/T7/.../data/spatial_split/`) through a trained model and compares the
prediction with the CDL ground truth.

The split (block=1024px, 70/15/15, seed=42) was materialized by
`scratchpad/materialize_split.py`: each block is a folder with `s2.tif`
(23 dates × 10 bands = 230 channels) + `cdl.tif`, under `train/ val/ test/`.
This notebook picks one **test** block — spatially disjoint from training —
so the demo shows generalization to unseen ground.

In [9]:
# Best model directory across scenarios
import os

best_model_root_dir = "/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs"
best_model_dir = {
    "single_date":
        {
            "segformer": "exp_single_date_segformer_20260630-041347",
            "dlv3plus_cbam": "exp_single_date_deeplabv3plus_cbam_20260630-042801"
        },
    "mt_ndvi":
        {
            "segformer": "exp_mt_base_segformer_20260630-044523",
            "dlv3plus_cbam": "exp_mt_base_deeplabv3plus_cbam_20260630-050013"
        },
    "gsi":
        {
            "segformer": "exp_gsi_segformer_20260630-061835",
            "dlv3plus_cbam": "exp_gsi_deeplabv3plus_cbam_20260630-063939"
        },
    "rf":
        {
            "segformer": "exp_rf_segformer_20260630-055524",
            "dlv3plus_cbam": "exp_rf_deeplabv3plus_cbam_20260630-060808"
        }
}

for scenario, models in best_model_dir.items():
    print(f"\n=== {scenario} ===")
    for model_name, run_dir in models.items():
        full_path = os.path.join(best_model_root_dir, run_dir)
        print(f"{model_name}:")
        print(full_path)


=== single_date ===
segformer:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_single_date_segformer_20260630-041347
dlv3plus_cbam:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_single_date_deeplabv3plus_cbam_20260630-042801

=== mt_ndvi ===
segformer:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_mt_base_segformer_20260630-044523
dlv3plus_cbam:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_mt_base_deeplabv3plus_cbam_20260630-050013

=== gsi ===
segformer:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/COLLEGE/Tugas Akhir/models/runs_202605/runs/exp_gsi_segformer_20260630-061835
dlv3plus_cbam:
/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gm

In [10]:
# Register this repo as `cropmap_pipeline` regardless of checkout dir name.
import os, sys, importlib.util, importlib.machinery
from pathlib import Path

REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
os.environ['MLFLOW_DISABLE_TELEMETRY'] = 'true'

_pkg = 'cropmap_pipeline'
if _pkg not in sys.modules:
    _spec = importlib.machinery.ModuleSpec(_pkg, None, is_package=True)
    _mod = importlib.util.module_from_spec(_spec)
    _mod.__path__ = [str(REPO)]
    sys.modules[_pkg] = _mod

from cropmap_pipeline import config as C
print('Repo:', REPO, '| classes:', C.NUM_CLASSES)

Repo: /Users/dikaizm/Documents/PROGRAMMING/ml-ai/research-crop-mapping-thesis/research-crop-mapping-geoai/cropmap-remote-sensing-exps | classes: 9


In [11]:
import json, glob
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

from cropmap_pipeline.stages.training.train_segmentation import build_model, evaluate_test_set
from cropmap_pipeline.stages.training.normalization import _per_channel_percentiles

SPLIT_DIR = Path('/Volumes/T7/research-crop-mapping-geoai/data/spatial_split')
SCENARIO  = 'gsi'          # single_date | mt_ndvi | gsi | rf
ARCH      = 'segformer'    # must match the checkpoint
THRESH    = 0.5
PATCH     = C.PATCH_SIZE
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
manifest = json.loads((SPLIT_DIR / 'blocks_manifest.json').read_text())
test_blocks = [b for b in manifest['blocks'] if b['split'] == 'test']
print('test blocks:', [Path(b['s2']).parent.name for b in test_blocks])

test blocks: ['block_r0_c1', 'block_r2_c1', 'block_r2_c5', 'block_r4_c0', 'block_r4_c5', 'block_r5_c3', 'block_r5_c4']


## 1. Load one test block (S2 stack + CDL)

In [3]:
BLK = test_blocks[0]                       # pick the first test block
blk_dir = SPLIT_DIR / Path(BLK['s2']).parent
with rasterio.open(SPLIT_DIR / BLK['s2']) as s:
    s2 = s.read().astype(np.float32)       # (230, H, W)
    band_names = list(s.descriptions)
with rasterio.open(SPLIT_DIR / BLK['cdl']) as s:
    cdl = s.read(1).astype(np.int32)
gt = C.REMAP_LUT[np.clip(cdl, 0, 255)].astype(np.int64)   # 0=bg, 1..8 crops
print(f'block {blk_dir.name}: S2 {s2.shape} | {len(band_names)} channels | crops present:',
      sorted(set(np.unique(gt)) - {0}))

block block_r0_c1: S2 (230, 1024, 1024) | 230 channels | crops present: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


## 2. Block RGB + ground truth

In [4]:
def bidx(name): return band_names.index(name)
def stretch(a):
    a = a.copy(); a[a == C.S2_NODATA] = np.nan
    lo, hi = np.nanpercentile(a, [2, 98])
    return np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1)

mid = band_names[len(band_names)//2].split('_')[1]   # a mid-year date
rgb = np.dstack([stretch(s2[bidx(f'{b}_{mid}')]) for b in ('B4','B3','B2')])
pal = plt.cm.tab10(np.linspace(0,1,len(C.CDL_CLASS_NAMES)))
cmap = ListedColormap([(.9,.9,.9,1)]+[tuple(c) for c in pal])
norm = BoundaryNorm(np.arange(-.5, C.NUM_CLASSES+.5), C.NUM_CLASSES)
fig,ax = plt.subplots(1,2,figsize=(12,6))
ax[0].imshow(rgb); ax[0].set_title(f'{blk_dir.name} RGB ({mid})'); ax[0].axis('off')
ax[1].imshow(gt, cmap=cmap, norm=norm, interpolation='nearest'); ax[1].set_title('CDL ground truth'); ax[1].axis('off')
ax[1].legend(handles=[Patch(facecolor='0.9',label='bg')]+[Patch(facecolor=pal[i],label=n) for i,n in enumerate(C.CDL_CLASS_NAMES.values())],
             bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## 3. Select the scenario's channels from the block

Selection channel names (`band_YYYYMMDD`) match the block's band descriptions directly, so map by name. (single_date/mt_ndvi would pick dates instead.)

In [5]:
if SCENARIO in ('gsi', 'rf'):
    sel = json.loads((SPLIT_DIR.parent / f'select_{SCENARIO}_direct_s{THRESH:g}.json').read_text())
    want = sel['union_channels']
else:
    want = band_names   # baselines: all channels (date sub-selection omitted in this demo)
name_to_i = {n: i for i, n in enumerate(band_names)}
chan_idx = [name_to_i[n] for n in want if n in name_to_i]
print(f'{SCENARIO}: {len(chan_idx)} / {len(band_names)} channels selected')

gsi: 155 / 230 channels selected


## 4. Normalize + tiled inference over the block

Per-band percentile stats from the training run (`norm_stats_percentile.npz`) applied per selected channel; the block is predicted tile-by-tile (256 px) and stitched.

In [13]:
CKPT = os.path.join(best_model_root_dir, best_model_dir["gsi"]["segformer"], "best_model.pth")
# per-band norm stats (band-major over S2_BAND_NAMES); expand to selected channels
ns = SPLIT_DIR / 'norm_stats_percentile.npz'
d = np.load(ns); lo_b, hi_b = d['lo'], d['hi']   # (10,)
band_of = lambda ci: C.S2_BAND_NAMES.index(band_names[ci].split('_')[0])
lo = np.array([lo_b[band_of(ci)] for ci in chan_idx], np.float32)
hi = np.array([hi_b[band_of(ci)] for ci in chan_idx], np.float32)
den = np.maximum(hi - lo, 1.0)

# checkpoints are wrapped dicts: {model_state_dict, architecture, in_channels, num_classes, ...}
_ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
if isinstance(_ck, dict) and 'model_state_dict' in _ck:
    state = _ck['model_state_dict']
    _arch = _ck.get('architecture', ARCH)
    _inch = _ck.get('in_channels', len(chan_idx))
    _ncls = _ck.get('num_classes', C.NUM_CLASSES)
else:  # bare state_dict
    state, _arch, _inch, _ncls = _ck, ARCH, len(chan_idx), C.NUM_CLASSES
assert _inch == len(chan_idx), f'channel mismatch: checkpoint {_inch} vs selected {len(chan_idx)} — check SCENARIO/THRESH'
model = build_model(_arch, _inch, _ncls).to(DEVICE)
model.load_state_dict(state); model.eval()
print(f'loaded {_arch} | in_channels={_inch} | epoch={_ck.get("epoch","?") if isinstance(_ck,dict) else "?"} '
      f'best_miou={_ck.get("best_miou","?") if isinstance(_ck,dict) else "?"}')

Cn, Hh, Ww = len(chan_idx), s2.shape[1], s2.shape[2]
pred = np.zeros((Hh, Ww), np.uint8)
with torch.no_grad():
    for y in range(0, Hh, PATCH):
        for x in range(0, Ww, PATCH):
            ph, pw = min(PATCH, Hh-y), min(PATCH, Ww-x)
            tile = s2[np.ix_(chan_idx, range(y,y+ph), range(x,x+pw))].astype(np.float32)
            tile[tile == C.S2_NODATA] = 0.0; tile[~np.isfinite(tile)] = 0.0
            tile = np.clip((tile - lo[:,None,None]) / den[:,None,None], 0, 1)
            pad = np.zeros((Cn, PATCH, PATCH), np.float32); pad[:, :ph, :pw] = tile
            out = model(torch.from_numpy(pad).unsqueeze(0).to(DEVICE)).argmax(1)[0].cpu().numpy()
            pred[y:y+ph, x:x+pw] = out[:ph, :pw]
print('inference done; pred classes:', sorted(set(np.unique(pred)) - {0}))

RuntimeError: Error(s) in loading state_dict for Segformer:
	Missing key(s) in state_dict: "encoder.patch_embed1.proj.weight", "encoder.patch_embed1.proj.bias", "encoder.patch_embed1.norm.weight", "encoder.patch_embed1.norm.bias", "encoder.patch_embed2.proj.weight", "encoder.patch_embed2.proj.bias", "encoder.patch_embed2.norm.weight", "encoder.patch_embed2.norm.bias", "encoder.patch_embed3.proj.weight", "encoder.patch_embed3.proj.bias", "encoder.patch_embed3.norm.weight", "encoder.patch_embed3.norm.bias", "encoder.patch_embed4.proj.weight", "encoder.patch_embed4.proj.bias", "encoder.patch_embed4.norm.weight", "encoder.patch_embed4.norm.bias", "encoder.block1.0.norm1.weight", "encoder.block1.0.norm1.bias", "encoder.block1.0.attn.q.weight", "encoder.block1.0.attn.q.bias", "encoder.block1.0.attn.kv.weight", "encoder.block1.0.attn.kv.bias", "encoder.block1.0.attn.proj.weight", "encoder.block1.0.attn.proj.bias", "encoder.block1.0.attn.sr.weight", "encoder.block1.0.attn.sr.bias", "encoder.bl

## 5. Prediction vs ground truth + per-crop IoU

In [7]:
if ckpts:
    def iou(gt, pr, k):
        i = ((gt==k)&(pr==k)).sum(); u = ((gt==k)|(pr==k)).sum(); return i/u if u else np.nan
    ious = {C.CDL_CLASS_NAMES[c]: iou(gt, pred, k+1) for k, c in enumerate(C.KEEP_CLASSES)}
    miou = np.nanmean([v for v in ious.values() if not np.isnan(v)])
    oa   = (gt == pred).mean()
    print(f'block {blk_dir.name}  mIoU {miou:.4f}  OA {oa:.4f}')
    for n, v in ious.items(): print(f'  {n:14s} IoU {v:.3f}')
    err = (gt != pred).astype(int)
    fig, ax = plt.subplots(1, 3, figsize=(16, 6))
    ax[0].imshow(gt,   cmap=cmap, norm=norm, interpolation='nearest'); ax[0].set_title('Ground truth')
    ax[1].imshow(pred, cmap=cmap, norm=norm, interpolation='nearest'); ax[1].set_title(f'Prediction ({SCENARIO}/{ARCH})')
    ax[2].imshow(err,  cmap='Reds', interpolation='nearest'); ax[2].set_title('Errors')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('No checkpoint — ran block load + viz only.')

No checkpoint — ran block load + viz only.
